# Smoking is bad

In [ ]:
# Packages are installed with pixi (see pixi.toml), so here we only import them
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

In [ ]:
DATA_PATH = Path.cwd() / "data" / "smoking_data.csv"
df = pd.read_csv(DATA_PATH, index_col=0)
df.head()

In [ ]:
# 1314 people and 5 columns
df.shape

In [ ]:
# By default .describe() only summarises the numeric columns (age, salary).
# With include="all", the text columns get count / unique / top / freq
# instead of mean and std: each has just two values (Alive/Dead, Yes/No,
# male/female). Salary has a count of only 614, so it has missing values.
df.describe(include="all")

In [ ]:
# Only salary has missing values (700 of 1314, about 53%).
# Salary is not needed to answer the smoking question, so we keep these rows
# instead of dropping them, and check later whether salary matters at all.
df.isna().sum()

In [ ]:
# Turn the text answers into 1/0 so that the mean of a column is a share
df["alive"] = (df["outcome"] == "Alive").astype(int)
df["smokes"] = (df["smoker"] == "Yes").astype(int)
df[["outcome", "alive", "smoker", "smokes"]].head()

## Smoking analysis
Let's first check what your cousin did.

In [ ]:
df.groupby(["smokes"]).agg(prob=("alive", np.mean))

You should get the same results.

# Is smoking really good?
Now it is up to you to run the analysis further and check what could be underlying your cousings results.

In [ ]:
# Ten-year age groups, used in all plots and in the age adjustment below.
# The robustness check at the end shows the result does not depend on this.
AGE_BINS = [18, 25, 35, 45, 55, 65, 75, 85]
AGE_LABELS = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84"]
df["age_group"] = pd.cut(
    df["age"], bins=AGE_BINS, labels=AGE_LABELS, right=False
)

# Readable group names and one fixed colour per group for every plot
df["group"] = df["smoker"].map({"No": "Non-smoker", "Yes": "Smoker"})
COLOURS = {"Non-smoker": "#1f77b4", "Smoker": "#ff7f0e"}
sns.set_theme(style="whitegrid")

df["age_group"].value_counts().sort_index()

## Why do smokers survive more often?

For age to explain the cousin's result, two things need to be true: age has to matter for survival, and smokers and non-smokers have to differ in age. The two plots below check both.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: survival by age group for everyone
survival_by_age = df.groupby("age_group", observed=True)["alive"].mean()
axes[0].plot(AGE_LABELS, survival_by_age.values, marker="o", color="dimgrey")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Survival drops sharply after middle age")
axes[0].set_xlabel("Age group")
axes[0].set_ylabel("Share alive")

# Right: age distribution of each group, in % so the different
# group sizes (732 non-smokers, 582 smokers) do not matter
sns.histplot(
    data=df,
    x="age",
    hue="group",
    hue_order=["Non-smoker", "Smoker"],
    palette=COLOURS,
    bins=range(18, 90, 6),
    stat="percent",
    common_norm=False,
    element="step",
    fill=False,
    linewidth=2,
    ax=axes[1],
)
axes[1].set_ylim(bottom=0)
axes[1].set_title("Smokers are younger: few of them are over 65")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("% of group")
axes[1].get_legend().set_title(None)

fig.tight_layout()
plt.show()

**What the plots show.** Until the mid-40s, more than 90% of people are still alive, but then survival falls fast: 61% at 55-64, 21% at 65-74 and nobody at 75-84. At the same time, 26% of non-smokers are 65 or older, compared with only 9% of smokers. So the non-smoker group holds far more of the people with the lowest survival, and that pulls its average down.

## Comparing people of the same age

In [ ]:
# Survival and group size for each age group, separately for both groups
by_age = (
    df.groupby(["age_group", "group"], observed=True)["alive"]
    .agg(share_alive="mean", n="size")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, part in by_age.groupby("group"):
    ax.plot(
        part["age_group"].astype(str),
        part["share_alive"],
        marker="o",
        linewidth=2,
        color=COLOURS[name],
    )

# Label the lines directly instead of using a legend
ax.text(
    2.1, 0.99, "Non-smokers",
    color=COLOURS["Non-smoker"], fontweight="bold",
)
ax.text(
    2.2, 0.72, "Smokers",
    color=COLOURS["Smoker"], fontweight="bold",
)

# Flag the small groups so the oldest points are not over-read
n_old = by_age.query("group == 'Smoker' and n < 50")["n"].tolist()
ax.text(
    4.1,
    0.92,
    f"Few smokers are over 65\n({n_old[0]} aged 65-74, "
    f"{n_old[1]} aged 75-84),\nso these points are less certain",
    fontsize=9,
    color="dimgrey",
)

ax.set_ylim(0, 1.05)
ax.set_title("At the same age, smokers are less likely to be alive")
ax.set_xlabel("Age group")
ax.set_ylabel("Share alive")
fig.tight_layout()
plt.show()

by_age

**Comparing like with like.** In five of the seven age groups, non-smokers are more likely to be alive. In the other two the groups are tied: almost everyone aged 25-34 survives, and nobody aged 75-84 does. The gap is largest in middle age, about 11 percentage points at 55-64 (66% vs 55%).

## Back to the cousin's question

In [ ]:
# Raw survival: what the cousin compared
raw = df.groupby("group")["alive"].mean()

# Age-adjusted survival: weight each age group's survival rate by how
# common that age group is in the whole sample, so both groups get the
# same age mix
survival_table = df.pivot_table(
    index="age_group",
    columns="group",
    values="alive",
    aggfunc="mean",
    observed=True,
)
age_weights = df["age_group"].value_counts(normalize=True)
adjusted = survival_table.mul(age_weights, axis=0).sum()

comparison = pd.DataFrame(
    {"Raw (cousin's comparison)": raw, "Same age mix": adjusted}
).T

fig, ax = plt.subplots(figsize=(7, 4.5))
comparison.plot.bar(
    ax=ax,
    color=[COLOURS["Non-smoker"], COLOURS["Smoker"]],
    rot=0,
    width=0.7,
    legend=False,
)
for bars in ax.containers:
    ax.bar_label(bars, fmt="{:.0%}", padding=3)

# Label the bars directly instead of using a legend
ax.text(
    -0.17, 0.05, "Non-smokers",
    color="white", fontweight="bold", rotation=90, ha="center",
)
ax.text(
    0.17, 0.05, "Smokers",
    color="white", fontweight="bold", rotation=90, ha="center",
)

ax.set_ylim(0, 1)
ax.grid(axis="x", visible=False)
ax.set_title("Give both groups the same age mix and the gap flips")
ax.set_ylabel("Share alive")
fig.tight_layout()
plt.show()

comparison.round(3)

**Putting it together.** The cousin's comparison mixes up two things: the effect of smoking and the fact that smokers are younger. If both groups had the same age mix as the whole sample, non-smokers would come out ahead, 74% vs 70%. So the higher raw survival of smokers comes from their age, not from smoking.